In [ ]:
!pip install dlib opencv-python imutils
!pip install diffusers transformers accelerate safetensors
!pip install gradio --quiet


In [ ]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2


--2026-02-18 04:04:53--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-02-18 04:04:54--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  43.9MB/s    in 1.4s    

2026-02-18 04:04:55 (43.9 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



In [ ]:
import cv2
import dlib
import torch
import numpy as np
from imutils import face_utils
from scipy.spatial import distance
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode
from diffusers import StableDiffusionPipeline


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
import dlib
from imutils import face_utils

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np

def take_photo_with_preview(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      // 1. Create the UI elements
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // 2. Add a Capture button
      const capture = document.createElement('button');
      capture.textContent = 'Capture Avatar Base';
      capture.style.background = 'orange';
      capture.style.color = 'white';
      capture.style.padding = '10px';
      div.appendChild(capture);

      // Resize the output frame
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // 3. Wait for the user to click Capture
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      // 4. Clean up the stream
      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
    // We immediately trigger the function we just defined
    takePhoto(%f);
    ''' % quality)

  display(js)
  # eval_js executes the LAST thing in the JS block, which is the takePhoto call
  data = eval_js("takePhoto(%f)" % quality)

  # Decode and save
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# --- RUN IT ---
try:
  img_path = take_photo_with_preview()
  print(f"Captured: {img_path}")

  # Verify with OpenCV
  img = cv2.imread(img_path)
  print(f"Image Resolution: {img.shape[1]}x{img.shape[0]}")
except Exception as e:
  print(f"Error: {e}")


  # 1. This is the 'detector' that finds the box around the face
detector = dlib.get_frontal_face_detector()

# 2. This is the 'predictor' that finds the 68 specific points (eyes, nose, etc.)
# IMPORTANT: You must have the 'shape_predictor_68_face_landmarks.dat' file in your folder
predictor_path = "shape_predictor_68_face_landmarks.dat"
predictor = dlib.shape_predictor(predictor_path)


<IPython.core.display.Javascript object>

KeyboardInterrupt: 

In [ ]:
def dist(a, b):
    """
    Calculates the Euclidean distance between two points (x, y).
    a and b should be numpy arrays or lists like [x, y].
    """
    return np.linalg.norm(a - b)

In [ ]:
def get_detailed_features(image, landmarks, face_box):
    # --- 1. FACE SHAPE & CHUBBY CHEEKS ---
    face_width = dist(landmarks[0], landmarks[16])
    # Measuring height from bridge of nose (27) to chin (8) for vertical ratio
    face_height = dist(landmarks[27], landmarks[8])
    face_ratio = face_height / face_width

    # Face Shape Logic
    if face_ratio > 0.92:
        face_shape_tag = "(elongated narrow face:1.5), (sharp angular jawline:1.3), thin face"
    elif face_ratio < 0.78:
        face_shape_tag = "(wide round face:1.4), (full soft cheeks:1.3)"
    else:
        face_shape_tag = "(balanced oval face:1.3)"

    # Cheek Detection (Chubby vs Sculpted)
    cheek_width = dist(landmarks[1], landmarks[15])
    if cheek_width / face_width > 0.95:
        cheek_tag = "(specifically full chubby cheeks:1.5), (rounded lower face:1.3)"
    else:
        cheek_tag = "(sculpted defined facial structure:1.2)"

    # --- 2. THE IMPROVED EYE ANALYSIS (Average Opening) ---
    # Using the averaged vertical opening for stable likeness
    left_eye_h = (dist(landmarks[37], landmarks[41]) + dist(landmarks[38], landmarks[40])) / 2
    right_eye_h = (dist(landmarks[43], landmarks[47]) + dist(landmarks[44], landmarks[46])) / 2
    avg_eye_h = (left_eye_h + right_eye_h) / 2

    eye_to_face_h = avg_eye_h / face_height

    if eye_to_face_h < 0.045:
        eye_size_tag = "(specifically small narrow eyes:1.6), (realistic squinty eyes:1.5), (natural human eye size:1.4)"
    elif eye_to_face_h > 0.075:
        eye_size_tag = "(large expressive eyes:1.3), (detailed iris:1.2)"
    else:
        eye_size_tag = "(medium natural almond eyes:1.3), (standard human eye ratio:1.2)"

    # --- 3. COLORS (Eye & Hair) ---
    # Eye color: sampling left iris
    eye_region = image[landmarks[38][1]:landmarks[40][1], landmarks[37][0]:landmarks[38][0]]
    if eye_region.size > 0:
        avg_bgr = eye_region.mean(axis=(0,1))
        # Simple BGR to Color Name mapping
        if avg_bgr[2] > avg_bgr[0] and avg_bgr[2] > avg_bgr[1]: eye_color = "warm brown"
        elif avg_bgr[0] > avg_bgr[2]: eye_color = "deep blue"
        else: eye_color = "hazel/dark"
    else:
        eye_color = "dark"

    # Hair color: sampling area above face box
    top_y = max(0, face_box.top() - 40)
    hair_sample = image[top_y:face_box.top(), face_box.left():face_box.right()]
    if hair_sample.size > 0:
        h_mean = hair_sample.mean()
        hair_color = "pitch black" if h_mean < 50 else "dark brown" if h_mean < 100 else "light/blonde"
    else:
        hair_color = "dark"

    # --- 4. BROWS, NOSE, & LIPS ---
    brow_arch = dist(landmarks[19], landmarks[27])
    brow_type = "high arched" if brow_arch > (face_width * 0.15) else "straight flat"

    nose_w = dist(landmarks[31], landmarks[35])
    nose_h = dist(landmarks[27], landmarks[33])
    nose_type = "refined button nose" if (nose_w / nose_h) < 0.9 else "broad strong nose"

    lip_thickness = dist(landmarks[51], landmarks[57])
    mouth_width = dist(landmarks[48], landmarks[54])
    lip_type = "full plush lips" if lip_thickness > (mouth_width * 0.45) else "thin elegant lips"

    # --- 6. CHIN & JAW DETECTION ---
    # Vertical distance from lower lip (57) to chin tip (8)
    chin_height = dist(landmarks[57], landmarks[8])
    # Horizontal distance between jaw points near the chin
    chin_width = dist(landmarks[7], landmarks[9])

    chin_to_face_ratio = chin_height / face_height

    # Chin Type Logic
    if chin_to_face_ratio > 0.18:
        chin_tag = "(prominent elongated chin:1.3), (defined lower jaw:1.2)"
    elif chin_to_face_ratio < 0.12:
        chin_tag = "(small recessed chin:1.3), (soft rounded jaw:1.2)"
    else:
        chin_tag = "(balanced proportional chin:1.1)"

    # Pointy vs Square Jaw
    if (chin_width / face_width) < 0.15:
        jaw_tag = "(v-shaped pointy chin:1.4)"
    else:
        jaw_tag = "(square strong jawline:1.3)"

    return face_shape_tag, eye_color, brow_type, hair_color, eye_size_tag, nose_type, lip_type, cheek_tag, chin_tag, jaw_tag

In [ ]:
def decide_skin_tone_improved(face_crop, landmarks_local):
    """
    Analyzes skin tone by sampling specific zones (cheeks/forehead)
    to avoid shadows, lips, and eyes.
    """
    try:
        # Convert BGR to RGB
        rgb_face = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        h, w, _ = rgb_face.shape

        # Define 3 specific "Pure Skin" zones (Cheeks and Forehead)
        # Zone 1: Forehead (Center)
        forehead = rgb_face[int(h*0.1):int(h*0.2), int(w*0.4):int(w*0.6)]
        # Zone 2: Right Cheek
        r_cheek = rgb_face[int(h*0.5):int(h*0.6), int(w*0.2):int(w*0.3)]
        # Zone 3: Left Cheek
        l_cheek = rgb_face[int(h*0.5):int(h*0.6), int(w*0.7):int(w*0.8)]

        # Combine samples
        combined_sample = np.concatenate((forehead.flatten(), r_cheek.flatten(), l_cheek.flatten()))
        avg_brightness = np.mean(combined_sample)

        # Map to specific Pixar-friendly skin descriptive strings
        if avg_brightness > 200:
            return "pale porcelain"
        elif avg_brightness > 170:
            return "fair"
        elif avg_brightness > 140:
            return "light beige"
        elif avg_brightness > 110:
            return "warm olive"
        elif avg_brightness > 70:
            return "rich tan"
        else:
            return "deep espresso"

    except Exception:
        return "neutral" # Fallback

In [ ]:
import transformers
# Fix for transformers 5.0 compatibility with diffusers
if not hasattr(transformers, "MT5Tokenizer"):
    from transformers import T5Tokenizer
    transformers.MT5Tokenizer = T5Tokenizer

# Now you can import diffusers without the RuntimeError
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler
import torch
model_id = "SG161222/RealVisXL_V4.0_Lightning"

pipe = AutoPipelineForText2Image.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 2. Use the SDE Karras scheduler for the smoothest "Pixar" skin
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="sde-dpmsolver++"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--RealVisXL_V4.0_Lightning/snapshots/39b41eff57c9d10c62119590901d6771863b426d/text_encoder_2
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--SG161222--RealVisXL_V4.0_Lightning/snapshots/39b41eff57c9d10c62119590901d6771863b426d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def generate_pixar_prompt(gender, age_cat, skin_tone, e_color, h_color, b_type, e_size, n_type, l_type, c_tag, f_shape, chin_tag, jaw_tag):

    gender_noun = "male" if gender.lower() == "male" else "female" # Using boy/girl often pushes more cartoonish proportions than man/woman

    # 1. THE "CARICATURE" BLOCK
    # We use 'stylized' and 'proportions' to tell the AI to break human rules.
    likeness_block = f"""
    (extremely stylized 3D character:1.4), (big head small body proportions:1.2),
    (smooth {f_shape} face:1.5), (tiny {n_type} nose:1.4),
    ({jaw_tag}:1.1), ({chin_tag}:1.1), ({l_type} mouth:1.3), {c_tag}
    """

    # 2. THE "TOY" AESTHETIC
    # Keywords like 'vinyl', 'sculpt', and 'solid' stop it from looking like a real human.
    style_lock = f"""
    (Pixar movie style:1.5), (massive expressive {e_size} eyes:1.3),
    (Disney 3D render:1.3), (chibi aesthetic:1.1),
    (solid stylized hair:1.4), clean bold shapes, high-end toy design
    """

    # 3. ANTI-REALISM RENDER
    # We remove '8k' and 'vivid' which can sometimes cause hyper-detail.
    render_quality = "(smooth matte vinyl skin:1.5), subsurface scattering, soft clay shaders, rim lighting, octane render, stylized lighting"

    # 4. FINAL ASSEMBLY
    prompt = f"""
    {age_cat} {gender_noun}, {likeness_block}, {style_lock},
    ({skin_tone} skin:1.2), ({e_color} iris:1.4), ({h_color} sculpted hair:1.3),
    {render_quality}, ({b_type} eyebrows:1.2), masterpiece, solid color background
    """

    # 5. AGGRESSIVE NEGATIVES
    # This is the most important part to stop it from looking like a photo.
    negative = """
    (photorealistic:1.8), (real life photo:1.8), (human skin:1.6),
    (realistic eyes:1.6), (skin pores:1.5), (wrinkles:1.5), (fine hair strands:1.4),
    makeup, lipstick, eyelashes, sharp cheekbones, skinny face,
    uncanny valley, messy, grainy, blurry, 2D, anime, sketches
    """

    return prompt, negative

In [ ]:
import gradio as gr

def process_and_generate(input_img, gender, age_cat):
    if input_img is None:
        return None

    try:
        # 1. Image Preparation
        image = cv2.cvtColor(np.array(input_img), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)

        if len(faces) == 0:
            return gr.Warning("No face detected! Please center your face in the camera.")

        # 2. Extract Landmarks & Proportions
        face = faces[0]
        shape = predictor(gray, face)
        landmarks = face_utils.shape_to_np(shape)

        # 3. Geometric & Color Analysis
        x1, y1, x2, y2 = face.left(), face.top(), face.right(), face.bottom()
        face_crop = image[max(0, y1):y2, max(0, x1):x2]

        skin_tone = decide_skin_tone_improved(face_crop, landmarks)
        (f_shape, e_color, b_type, h_color,
        e_size, n_type, l_type, c_tag, chin_tag, jaw_tag) = get_detailed_features(image, landmarks, face)

        # 4. Generate the Precise Prompt
        # Passing 'age_cat' directly into the prompt generator
        p, n = generate_pixar_prompt(
            gender, age_cat, skin_tone,
            e_color, h_color, b_type,
            e_size, n_type, l_type, c_tag, f_shape,
            chin_tag, jaw_tag
        )

        # 5. Inference
        # Using a fixed seed for consistency or None for variety
        result = pipe(
            prompt=p,
            negative_prompt=n,
            num_inference_steps=8,
            guidance_scale=3.5 # Slightly higher for more "Pretty" stylization
        ).images[0]

        detection_details = f"""
        ### 🧬 Detected Human DNA:
        - Face: {f_shape.title()} | {jaw_tag} | {chin_tag}
        - Features: {n_type.replace('_', ' ')} nose | {l_type} lips
        - Colors: {skin_tone} skin | {e_color} eyes | {h_color} hair
        """

        return result, detection_details

    except Exception as e:
        print(f"Error in processing: {e}")
        return None

# --- UPDATED GRADIO LAYOUT ---

with gr.Blocks(theme=gr.themes.Soft(), css=".container { max-width: 800px; margin: auto; }") as demo:
    gr.Markdown("# ✨ 3D Pretty Clay Avatar Studio")
    gr.Markdown("Capture your photo to create a high-end 3D character with your exact features.")

    with gr.Row():
        # Input Column
        with gr.Column(scale=1):
            input_cam = gr.Image(sources=["webcam"], type="pil", label="Capture your face")

            with gr.Row():
                gender_select = gr.Radio(["male", "female"], label="Gender", value="female")
                # Requirement: Age input added
                age_select = gr.Dropdown(
                    ["Kid", "Teenage", "Youth", "Adult"],
                    label="Age Category",
                    value="Youth"
                )

            btn = gr.Button("🎨 Create Disney Avatar", variant="primary")

        # Output Column (Now only showing the result)
        with gr.Column(scale=1):
            avatar_result = gr.Image(label="Your 3D Avatar", interactive=False, elem_id="output-img")

            # CHANGE THIS LINE: Create a component that can be updated
            status_display = gr.Markdown("### ✨ Likeness Details Locked:\n*Waiting for capture...*")

    # Click Event: Note only one output now
    btn.click(
            fn=process_and_generate,
            inputs=[input_cam, gender_select, age_select],
            outputs=[avatar_result, status_display]
        )

if __name__ == "__main__":
    demo.queue().launch(debug=True)

/tmp/ipython-input-99593076.py:62: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=".container { max-width: 800px; margin: auto; }") as demo:
/tmp/ipython-input-99593076.py:62: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=".container { max-width: 800px; margin: auto; }") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://faffa866303e1839b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Error in processing: name 'predictor' is not defined


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2202, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1924, in postprocess_data
    self.validate_outputs(block_fn, predictions)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1879, in validate_outputs
    rai

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://faffa866303e1839b4.gradio.live
